<span STYLE="font-size:150%"> 
    Segment microCT scans
</span>

Docker image: gnasello/slicer-env:2023-07-06 \
Latest update: 10 March 2023

- load image stack in Slicer
- segment mineralized tissue
- compute segmented statistics (volumes)

# Install libraries

Run the command below one time only, then comment it

In [ ]:
# !git clone https://github.com/gabnasello/pyslicer.git
# !PythonSlicer -m pip install -e pyslicer/
# !PythonSlicer -m pip install matplotlib

# import os
# os._exit(00)

# Load libraries

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Read `calibration_stats.csv` file

In [ ]:
# this cell is tagged 'parameters'
output_dir = 'microCT_calibration'
calibration_file =  output_dir + '/calibration_stats.csv'
output_greyvalues_file = 'segmented_volumes/segments_greyvalues.csv'

In [ ]:
output_dir_path = Path(output_dir).resolve()
calibration_file_path = Path(calibration_file).resolve()

In [ ]:
df = pd.read_csv(calibration_file)
df

# Prepare Data + Linear Regression

In [ ]:
# Ensure numeric columns
df['Density_g_cm3'] = pd.to_numeric(df['Density_g_cm3'], errors='coerce')
df['Mean'] = pd.to_numeric(df['Mean'], errors='coerce')

# Extract variables
x = df['Mean'].values
y = df['Density_g_cm3'].values

# Linear regression
z = np.polyfit(x, y, 1)
slope, intercept = z

print(f"Slope:     {slope:.6e}")
print(f"Intercept: {intercept:.6f}")
print(f"Equation:  y = {slope:.6e} * x + {intercept:.6f}")

# Plot Linear Regression

In [ ]:
fig1, ax1 = plt.subplots(figsize=(5, 5))

# Optional axis limits — adjust if needed
ax1.set_xlim(0, 2**16-1)
ax1.set_ylim(0, 5)

# Scatter plot
ax1.scatter(x, y, s=100, c='blue', edgecolors='black')

# Regression line
x_trend = np.linspace(x.min(), x.max(), 200)
y_trend = slope * x_trend + intercept
ax1.plot(x_trend, y_trend, "r--", label="Linear Fit")

# Labels
ax1.set_title("Density vs Grayscale (Mean)")
ax1.set_xlabel("Grayscale Value")
ax1.set_ylabel("Density (g/cm³)")
ax1.grid(True, linestyle='--', alpha=0.7)

# Equation text
ax1.text(x.min(), y.max(), f"y = {slope:.2e} x + {intercept:.3f}", fontsize=12)

plt.show()

# Compute grayscale value for a given target density

In [ ]:
# Choose your target density
target_density = 1.5   # g/cm3

# Solve for grayscale:  x = (y - intercept) / slope
target_grayscale = (target_density - intercept) / slope

print(f"Target Density:   {target_density}")
print(f"Target Grayscale: {target_grayscale:.3f}")

# Plot Regression + Target Point

In [ ]:
fig2, ax2 = plt.subplots(figsize=(5, 5))

# Scatter
ax2.scatter(x, y, s=100, c='blue', edgecolors='black')

# Regression line
ax2.plot(x_trend, y_trend, "r--", label="Linear Fit")

# Target point
ax2.scatter([target_grayscale], [target_density],
            color='red', s=120, zorder=5)

# Connector lines (black)
ax2.plot([target_grayscale, target_grayscale],
         [0, target_density], color='black', linestyle='-', alpha=0.7, linewidth=1)

ax2.plot([0, target_grayscale],
         [target_density, target_density], color='black', linestyle='-', alpha=0.7, linewidth=1)

# Annotation
ax2.annotate(
    f"Target\n({target_grayscale:.1f}, {target_density})",
    (target_grayscale, target_density),
    xytext=(-90, 40),
    textcoords='offset points',
    fontsize=12,
    arrowprops=dict(arrowstyle='->', color='red')
)

# Titles and labels
ax2.set_title("Density vs Grayscale with Target Density Point")
ax2.set_xlabel("Grayscale Value")
ax2.set_ylabel("Density (g/cm³)")
ax2.grid(True, linestyle='--', alpha=0.7)

# ----------------------------------------
# MATCH AXES LIMITS FROM FIRST PLOT
# ----------------------------------------
ax2.set_xlim(ax1.get_xlim())
ax2.set_ylim(ax1.get_ylim())

plt.show()

In [ ]:
fig2.savefig(output_dir_path / "GrayscaleValue_vs_Density_plot.png", dpi=300)
print("Saved as GrayscaleValue_vs_Density_plot.png")

# Save calibrated thresholding value

In [ ]:
segments_greyvalues = {
    "Bone": [target_grayscale, 65535],
    }

segments_greyvalues

In [ ]:
grey_df = pd.DataFrame(segments_greyvalues)

grey_df.to_csv(output_greyvalues_file, index=False)